In [1]:
# Funcao get_certainty so funciona pra deepseek, ajustar para phi4

In [4]:
import os
from datasets import load_dataset
from vllm import LLM, SamplingParams
import torch
from transformers import PreTrainedTokenizer
from typing import List, Union
from pathlib import Path
import numpy as np
from transformers import LogitsProcessorList
from transformers import AutoTokenizer
import json
import time
import pandas as pd


In [5]:
# Select model
# model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
model_id = "microsoft/Phi-4-reasoning-plus"
# model_id = "Qwen/Qwen3-14B"
# model_id = "Qwen/Qwen3-0.6B"

In [6]:
# Select where the model will be stored (optional)
download_dir = "/home/jovyan/runai2new/models/"

In [7]:
# Select task, seed, temperature, and top_p
TASK = "aime2025"
SEED = 42
TEMPERATURE = 0.6
TOP_P = 0.95 
MAX_TOKENS = 32768

In [8]:
if TASK=="aime2024":
    dataset = load_dataset("HuggingFaceH4/aime_2024")
    problems = dataset['train']['problem']
    answers = dataset['train']['answer']
    answers = [answer if answer[0] != '0' else answer[1:] for answer in answers] # IF THE ANSWER HAS TWO DIGITS, REMOVE THE LEADING ZERO
elif TASK=="aime2025":
    dataset = load_dataset("MathArena/aime_2025")
    problems = dataset['train']['problem']
    answers = dataset['train']['answer']
    answers = [str(answer) for answer in answers] # STORE ANSWERS AS STRINGS FOR CONSISTENCY WITH AIME2024
else:
    print("The selected task doesn't exist")

In [9]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [10]:
llm = LLM(
    model=model_id,
    download_dir=download_dir,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.95,
)

INFO 09-15 08:42:26 [config.py:1604] Using max model len 32768
INFO 09-15 08:42:26 [config.py:2434] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 09-15 08:42:27 [core.py:572] Waiting for init message from front-end.
INFO 09-15 08:42:27 [core.py:71] Initializing a V1 LLM engine (v0.10.0) with config: model='microsoft/Phi-4-reasoning-plus', speculative_config=None, tokenizer='microsoft/Phi-4-reasoning-plus', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir='/home/jovyan/runai2new/models/', load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning

Loading safetensors checkpoint shards:   0% Completed | 0/6 [00:00<?, ?it/s]


INFO 09-15 08:42:35 [default_loader.py:262] Loading weights took 5.13 seconds
INFO 09-15 08:42:36 [gpu_model_runner.py:1892] Model loading took 27.3915 GiB and 6.283819 seconds
INFO 09-15 08:42:42 [backends.py:530] Using cache directory: /home/jovyan/.cache/vllm/torch_compile_cache/c9796037ac/rank_0_0/backbone for vLLM's torch.compile
INFO 09-15 08:42:42 [backends.py:541] Dynamo bytecode transform time: 6.34 s
INFO 09-15 08:42:47 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 4.542 s
INFO 09-15 08:42:51 [monitor.py:34] torch.compile takes 6.34 s in total
INFO 09-15 08:42:52 [gpu_worker.py:255] Available KV cache memory: 39.48 GiB
INFO 09-15 08:42:52 [kv_cache_utils.py:833] GPU KV cache size: 206,976 tokens
INFO 09-15 08:42:52 [kv_cache_utils.py:837] Maximum concurrency for 32,768 tokens per request: 6.32x


Capturing CUDA graph shapes: 100%|██████████| 67/67 [00:03<00:00, 21.01it/s]


INFO 09-15 08:42:56 [gpu_model_runner.py:2485] Graph capturing finished in 4 secs, took 0.80 GiB
INFO 09-15 08:42:56 [core.py:193] init engine (profile, create kv cache, warmup model) took 20.13 seconds


In [11]:
tokenizer = llm.get_tokenizer()

In [12]:
params = SamplingParams(
    max_tokens=MAX_TOKENS,
    stop_token_ids=[tokenizer.eos_token_id],
    seed=SEED,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    skip_special_tokens=False,
    logprobs=1 # Get the top logprobs
)

In [13]:
# Create prompts from the questions
def create_prompt(question):
    messages = [
        {"role": "system", "content": "You are Phi, a language model trained by Microsoft to help users. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracing, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> {Thought section} </think> {Solution section}. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any errors, and revisiting previous steps. In the Solution section, based on various attempts, explorations, and reflections from the Thought section, systematically present the final solution that you deem correct. The Solution section should be logical, accurate, and concise and detail necessary steps needed to reach the conclusion. Now, try to solve the following question through the above guidelines:"},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return prompt

In [14]:
problems[0]

'Find the sum of all integer bases $b>9$ for which $17_b$ is a divisor of $97_b.$'

In [15]:
prompts = [create_prompt(problem) for problem in problems]

In [ ]:
batch_size = 5


for seed in range(1, 20):

    scores = []
    number_of_thinking_tokens = []
    
    # Create folder where the results will be stored
    Path(f"./certainties-{TASK}_temperature_{str(TEMPERATURE).replace('.','_')}_seed_{seed}-{model_id.replace('/','--')}/").mkdir(parents=True, exist_ok=True)
    path = Path(f"./certainties-{TASK}_temperature_{str(TEMPERATURE).replace('.','_')}_seed_{seed}-{model_id.replace('/','--')}/")

    
    with open(path/"certainties.txt","w") as f:
        f.write("SAMPLE,SCORE,NUMBER_OF_THINKING_TOKENS,TRUE_ANSWER,PREDICTED_ANSWER,CERTAINTY\n")
        
    with open(path/"summary.txt", "w") as f:
        f.write("NUMBER_OF_CORRECT_PREDICTIONS,PERCENTAGE_OF_CORRECT_PREDICTIONS,TOTAL_NUMBER_OF_THINKING_TOKENS,AVERAGE_NUMBER_OF_THINKING_TOKENS\n")
        
    for batch_start in range(0, len(prompts), batch_size):
        
        batch = prompts[batch_start:batch_start+batch_size]
        batch_outputs = llm.generate(batch, sampling_params=params)
        
        for idx, output in enumerate(batch_outputs):
            
            # certainty, pred_answer_tokens, result_tokens, candidate_answer = get_certainty(output)
            certainty = get_certainty(output)
            # print(result_tokens)
            sample_idx = batch_start+idx
            true_answer = answers[sample_idx]
            candidate_answer_raw = output.outputs[0].text
            
            with open(path/f"thoughts_{sample_idx}.txt", "w") as f:
                f.write(candidate_answer_raw)
                
            thinking_tokens = candidate_answer_raw.split("</think>")[0]
            num_thinking_tokens = len(tokenizer.encode(thinking_tokens))
            candidate_answer = candidate_answer_raw.split("boxed{")[-1].split("}")[0]
            # candidate_answer = pred_answer
            score = candidate_answer == true_answer
            number_of_thinking_tokens.append(num_thinking_tokens)
            scores.append(score)
            
            with open(path/"certainties.txt","a") as f:
                f.write(f"{sample_idx},{score},{num_thinking_tokens},{true_answer},{candidate_answer[:10]},{certainty}\n")
                # f.write(f"{sample_idx},{score},{num_thinking_tokens},{true_answer},{candidate_answer},{certainty}, {result_tokens}\n")
                
    with open(path/"summary.txt", "a") as f:
        f.write(f"{sum(scores)},{100*sum(scores)/len(scores)},{sum(number_of_thinking_tokens)},{sum(number_of_thinking_tokens)/len(scores)}")

Adding requests:   0%|          | 0/5 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/5 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [29]:
for idx, output in enumerate(batch_outputs):

    print(idx)
    logprobs = output.outputs[0].logprobs
    candidate_answer_raw = output.outputs[0].text

    certainty, pred_answer_tokens, result_tokens, candidate_answer = get_certainty(output)
    
    print(candidate_answer)
    
    # print(logprobs[-8
    #       :])
    # print()
    # print(candidate_answer_raw[-50:])
    # print(candidate_answer_raw)

0

1

2

3

4



In [ ]:
batch_outputs

In [ ]:
for idx, output in enumerate(batch_outputs):

    certainty, numeric_tokens, full_tokens, answer = get_certainty(output)
    print("Answer:", answer)
    print("Numeric tokens:", numeric_tokens)
    print("Full tokens:", full_tokens)


In [ ]:
np.exp(np.min([list(logprob.values())[0].logprob for logprob in logprobs][:k]))

In [ ]:
np.exp(-07)


In [ ]:
len(logprobs)

In [16]:
# Fix certainties
import csv 
seed = 3
for seed in range(64):
    # path = Path(f'reasoning_traces/reasoning_traces-aime2025_temperature_0_6_seed_{seed}-deepseek-ai--DeepSeek-R1-Distill-Qwen-14B')
    # path = Path(f'results/32k/reasoning_traces-aime2025_temperature_0_6_seed_{seed}-deepseek-ai--DeepSeek-R1-Distill-Llama-70B')
    path = Path(f"./certainties-{TASK}_temperature_{str(TEMPERATURE).replace('.','_')}_seed_{seed}-{model_id.replace('/','--')}/")

    file_path = f'{path}/certainties.txt'
    
    
    input_file = file_path
    output_file = f'{path}/certainties.txt'
    
    fixed_rows = []
    
    # Step 1: Read and fix rows
    with open(input_file, "r", encoding="utf-8") as infile:
        reader = csv.reader(infile)
        header = next(reader)  # Read header
        fixed_rows.append(header)
    
        for row in reader:
            if len(row) > 6:
                # Fix rows where a field includes commas
                joined_value = ','.join(row[4:-1])
                fixed_row = row[:4] + [joined_value] + [row[-1]]
                fixed_rows.append(fixed_row)
            else:
                fixed_rows.append(row)
    
    # Step 2: Write the fixed rows to a new file
    with open(output_file, "w", newline='', encoding="utf-8") as outfile:
        writer = csv.writer(outfile)
        writer.writerows(fixed_rows)

# Step 3: Optional – Load into pandas for verification
# df = pd.DataFrame(fixed_rows[1:], columns=fixed_rows[0])

# Convert numeric columns if needed
# df["NUMBER_OF_THINKING_TOKENS"] = pd.to_numeric(df["NUMBER_OF_THINKING_TOKENS"], errors="coerce")
# df["CERTAINTY"] = pd.to_numeric(df["CERTAINTY"], errors="coerce")

    print(f"Cleaned file saved as: {output_file}")


Cleaned file saved as: certainties-aime2024_temperature_0_6_seed_0-deepseek-ai--DeepSeek-R1-Distill-Qwen-14B/certainties.txt
Cleaned file saved as: certainties-aime2024_temperature_0_6_seed_1-deepseek-ai--DeepSeek-R1-Distill-Qwen-14B/certainties.txt
Cleaned file saved as: certainties-aime2024_temperature_0_6_seed_2-deepseek-ai--DeepSeek-R1-Distill-Qwen-14B/certainties.txt
Cleaned file saved as: certainties-aime2024_temperature_0_6_seed_3-deepseek-ai--DeepSeek-R1-Distill-Qwen-14B/certainties.txt
Cleaned file saved as: certainties-aime2024_temperature_0_6_seed_4-deepseek-ai--DeepSeek-R1-Distill-Qwen-14B/certainties.txt
Cleaned file saved as: certainties-aime2024_temperature_0_6_seed_5-deepseek-ai--DeepSeek-R1-Distill-Qwen-14B/certainties.txt
Cleaned file saved as: certainties-aime2024_temperature_0_6_seed_6-deepseek-ai--DeepSeek-R1-Distill-Qwen-14B/certainties.txt
Cleaned file saved as: certainties-aime2024_temperature_0_6_seed_7-deepseek-ai--DeepSeek-R1-Distill-Qwen-14B/certainties.txt


FileNotFoundError: [Errno 2] No such file or directory: 'certainties-aime2024_temperature_0_6_seed_53-deepseek-ai--DeepSeek-R1-Distill-Qwen-14B/certainties.txt'

In [21]:
#  Check cert

for seed in range(53):
    path = Path(f"./certainties-{TASK}_temperature_{str(TEMPERATURE).replace('.','_')}_seed_{seed}-{model_id.replace('/','--')}/")
            # with open(path/"certainties.txt","a") as f:
    print("Seed:", seed)
    df = pd.read_csv(path/"certainties.txt")

    for i in range(30):

        cert = df['CERTAINTY'][i]
        print(i)
        if cert > 0 and cert < 0.99: 
            print(cert)
    
    

Seed: 0
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
Seed: 1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
Seed: 2
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
Seed: 3
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
Seed: 4
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
Seed: 5
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
Seed: 6
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
Seed: 7
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
Seed: 8
0
1
2
3
0.9881530353715238
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
Seed: 9
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
Seed: 10
0
1
2
3
0.9881530353715238
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27

TypeError: '>' not supported between instances of 'str' and 'int'